# v82: Bayes decision hedged over every ranking family

**Version:** v82  
**Kaggle submission:** `submission_v82_hedge.csv`, built 27 Nov (see `../SUBMISSIONS.md` for whether it was uploaded)

**Question.** The failure rankings disagree a lot on the test depot (CatBoost against the v12 classifier: Spearman 0.58), and neither validation nor the locked hold-out nor the public scores can say which is right. The safest decision averages over all of them. This adds two ranking families to v66's: CatBoost without s120/s122, alone and averaged with the survival model (`src/v72_newmodels.py`, `v73_catboost.py`), and survival models trained with failures weighted by sqrt(repair cost) (`src/v80_costweight.py`, `v81_aftcw.py`).

**The harness.** A family of hypotheses about the test depot (which classifier ranking is right, how failure odds tilt with wear, overall level, repair-cost level) is weighted by the exact likelihood of the public scores of our earlier uploads. The decision then minimises posterior-expected cost. Every hypothesis is weighted only by the public-score likelihood (no extra prior).

**Artifacts used:** `reports/v18_qm.npy`, `v18_resid.npy`, `v19_q.npy`, `v21_H.npy`, `v22_q.npy`, `v24_q.npy`, `v24_pu_x2_q.npy`, `v18_ens10_q.npy`, `v18_ens_q.npy`, `p_oof_v2.npy`, `reports/v61_q.npz`, `v65_q.npz`, `v73_q.npz`, `v81_q.npz`, and `reports/v82all_post.csv` (27 public scores; `python src/v66_harness.py`, `v74_harness_cat.py`, `v82_harness_aftcw.py`, `v82_hedge.py`). Repair costs as in v51. GP class from v33 (by Daksh183).

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

In [3]:
REP, SUB = Path("../reports"), Path("../submissions")
qv12, m = np.load(REP / "v18_qm.npy"); resid = np.load(REP / "v18_resid.npy")
sig = lambda v: 1 / (1 + np.exp(-v)); logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
def zs(x): x = np.log1p(np.nan_to_num(np.maximum(x, 0), nan=np.nanmedian(x))); return (x - x.mean()) / x.std()
v19 = np.load(REP / "v19_q.npy"); H21 = np.load(REP / "v21_H.npy")
te_map = te.copy()
for col in ("s120", "s122"):                                                        # quantile-map the two test sensors onto train
    r = te[col].rank(pct=True); te_map[col] = np.nanquantile(tr[col], r.fillna(0.5).clip(0, 1)); te_map.loc[te[col].isna(), col] = np.nan
mdl = xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure)
q_map = Platt().fit(np.load(REP / "p_oof_v2.npy"), y).predict(mdl.predict_proba(te_map[S])[:, 1])
RANK = {"v12": qv12, "nos122": np.load(REP / "v22_q.npy"), "pu": np.load(REP / "v24_q.npy"), "pu_x2": np.load(REP / "v24_pu_x2_q.npy"),
        "ens10": np.load(REP / "v18_ens10_q.npy"), "ens4": np.load(REP / "v18_ens_q.npy"), "fix120": v19[0], "fix": v19[1],
        "nodrift": H21[2], "map": q_map}                                            # ten ranking hypotheses
RANK = {k: logit(np.ravel(v)[-len(te):]) for k, v in RANK.items()}
Z = {"none": np.zeros(len(te)), **{s: zs(te[s].to_numpy()) for s in ("s120", "s122", "s126", "s149")}}   # wear scores

def prob(r):
    """failure probabilities of one 'truth': blend of rankings, wear tilts, confidence slope b, level a"""
    g = lambda k, d: r[k] if k in r and isinstance(r[k], (int, float, str)) and r[k] == r[k] else d
    w2, R2, c2, z2 = g("w2", 0.0), g("R2", "v12"), g("c2", 0.0), g("z2", "none")
    L = (1 - r["w"] - w2) * RANK["v12"] + r["w"] * RANK[r["R"]] + w2 * RANK[R2]
    return r["fl0"] + (1 - r["fl0"]) * sig(r["a"] + r["b"] * L + r["c"] * Z[r["z"]] + c2 * Z[z2])
ALPHAS = (1.0, 1.15, 1.3, 1.45); Kd = 99; grid = (np.arange(Kd) + .5) / Kd          # repair-cost level multipliers, 99 cost draws

In [4]:
def posterior_means(T):
    """posterior-weighted failure probability and weight, per cost level alpha"""
    T = T.copy(); T.wt /= T.wt.sum()
    PB = {al: sum((r.wt * prob(r._asdict()) for r in T[T.alpha == al].itertuples()), np.zeros(len(te))) for al in ALPHAS}
    return PB, {al: T[T.alpha == al].wt.sum() for al in ALPHAS}

def bayes_decide(PB, Wa, draws):
    """flag and budget that minimise posterior-expected cost; budgets capped at 8,000 (the largest repair cost seen)"""
    MX = {al: np.clip(draws * al, 5, 8000) for al in ALPHAS}
    wsum = sum(Wa.values()); p = sum(PB.values()) / wsum
    Mx = np.concatenate([MX[al] for al in ALPHAS if Wa[al] > 0], 1)
    wcol = np.concatenate([PB[al][:, None] / np.maximum(p[:, None] * wsum, 1e-12) * np.ones((1, Kd)) / Kd for al in ALPHAS if Wa[al] > 0], 1)
    o = np.argsort(Mx, 1); Ms = np.take_along_axis(Mx, o, 1); cw = np.cumsum(np.take_along_axis(wcol, o, 1), 1)
    Q = 1 - (1 - Q_STAR) / np.maximum(p, 1e-9)
    idx = np.clip((cw < np.clip(Q, 0, 1)[:, None]).sum(1), 0, Ms.shape[1] - 1)
    b = np.minimum(np.where(Q > 0, Ms[np.arange(len(p)), idx], 0.0), 8000)
    fail = ((UNDER * np.maximum(Mx - b[:, None], 0) + OVER * np.maximum(b[:, None] - Mx, 0)) * wcol).sum(1); mu = (Mx * wcol).sum(1)
    fl = (INSPECT + p * fail + (1 - p) * OVER * b < p * mu).astype(int); return fl, np.where(fl == 1, b, 0.0)

def load_posterior(path):
    P = pd.read_csv(REP / path)
    print(f"{len(P)} truths; weight by wear tilt:", P.groupby("z").wt.sum().round(3).to_dict())
    return P[P.wt > 1e-4]

In [5]:
"""Small additive Gaussian process for ordered sensor histograms, research only."""
import numpy as np
from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize
from scipy.spatial.distance import cdist

START = (23,37,54,80,129,154)

def extract(data):
    groups=[]
    for start in START:
        width=9 if start==54 else 10
        x=data[[f"s{i:03d}" for i in range(start,start+width)]].to_numpy(float)
        total=x.sum(axis=1)
        ok=np.isfinite(x).all(1)&(x>=0).all(1)&(total>0)
        frac=x/np.where(ok,total,np.nan)[:,None]
        # Shape and within-histogram location both vary continuously.
        groups.append(np.column_stack([np.sqrt(frac),frac.cumsum(1)[:,:-1]]))
        if start==80: exposure=np.log1p(np.where(ok,total,np.nan))[:,None]
    return groups+[exposure]

def select(data,rows):
    return [x[rows] for x in data]

class GaussianSeverity:
    def blocks(self, features):
        result=[]
        for x,base,scale in zip(features,self.training,self.scale):
            distance=cdist(x,base,'sqeuclidean')
            result.append(np.nan_to_num(np.exp(-distance/(2*scale)),nan=0.))
        return np.asarray(result)

    def fit(self,features,y):
        self.training=features
        self.scale=[]
        ind=np.triu_indices(len(y),1)
        for x in features:
            ds=cdist(x,x,'sqeuclidean')[ind]
            ds=ds[np.isfinite(ds)]
            md=np.median(ds) if len(ds) else 1.
            self.scale.append(max(md,1.e-6))
        blocks=self.blocks(features)
        self.center=np.mean(y)
        z=y-self.center
        unit=np.eye(len(y))
        def value(logtheta,final=False):
            theta=np.exp(logtheta)
            cov=np.einsum('i,ijk->jk',theta[:-1],blocks)+theta[-1]*unit
            factor=cho_factor(cov,lower=True,check_finite=False)
            alpha=cho_solve(factor,z,check_finite=False)
            if final: return alpha
            loss=.5*z@alpha+np.log(np.diag(factor[0])).sum()+.5*(logtheta@logtheta)/4
            inv=cho_solve(factor,unit,check_finite=False)-np.outer(alpha,alpha)
            grad=.5*theta*np.r_[np.einsum('ij,kij->k',inv,blocks),np.trace(inv)]+logtheta/4
            return loss,grad
        res=minimize(value,np.zeros(len(blocks)+1),jac=True,method='L-BFGS-B',
                     bounds=[(-5.,5.)]*len(blocks)+[(-4.,3.)],options={'maxiter':35})
        self.amplitude=np.exp(res.x[:-1])
        self.alpha=value(res.x,True)
        self.bounds=(min(y),max(y))
        return self

    def predict(self,features):
        out=np.full(len(features[0]),np.nan)
        for start in range(0,len(out),800):
            rows=slice(start,start+800)
            vals=select(features,rows)
            valid=sum(np.isfinite(x).all(1).astype(int) for x in vals[:-1])>=3
            blocks=self.blocks(vals)
            pred=np.einsum('i,ijk->jk',self.amplitude,blocks)@self.alpha+self.center
            out[rows]=np.where(valid,np.clip(pred,*self.bounds),np.nan)
        return out

In [6]:
class GPAll(GaussianSeverity):
    """predicts every row; which rows count is decided below, the same way for both fits"""
    def predict(self, features):
        return np.clip(np.einsum("i,ijk->jk", self.amplitude, self.blocks(features)) @ self.alpha + self.center, *self.bounds)

f = tr[tr.failure == 1].reset_index(drop=True); ly = np.log(f.repair_cost.to_numpy())
START6 = (23, 37, 54, 80, 129, 154); KEEP = (37, 54, 80, 154)
SENS = ["s116", "s142", "s105", "s104", "s078"]
def z(d, cols):                                   # log1p, standardised with the training failures' mean and sd
    R = np.log1p(np.maximum(f[cols].to_numpy(float), 0))
    return (np.log1p(np.maximum(d[cols].to_numpy(float), 0)) - np.nanmean(R, 0)) / (np.nanstd(R, 0) + 1e-9)
def feats(d, wear):
    H = extract(d)
    return [H[START6.index(a)] for a in KEEP] + [z(d, wear), z(d, SENS)]
# rows the Gaussian process may price (as in v42): enough of the original blocks observed
valid = sum(np.isfinite(x).all(1).astype(int) for x in extract(f)[:6]) >= 3
sup = sum(np.isfinite(x).all(1).astype(int) for x in extract(te) + [z(te, ["s126", "s149", "s123"])]) >= 3
oofs, sevs = [], []
for wear in (["s126", "s149", "s123"], ["s126", "s149"]):
    X = feats(f, wear); o = np.full(len(ly), np.nan)
    for a, b in KFold(5, shuffle=True, random_state=42).split(ly):
        o[b] = GPAll().fit(select(X, a), ly[a]).predict(select(X, b))
    o[~valid] = np.nan; s = GPAll().fit(X, ly).predict(feats(te, wear)); s[~sup] = np.nan
    oofs.append(o); sevs.append(s)
oof, sev = (oofs[0] + oofs[1]) / 2, (sevs[0] + sevs[1]) / 2
ok = np.isfinite(oof); gres = ly[ok] - oof[ok]; sup = np.isfinite(sev)
print(f"GP out-of-fold R2 {1 - (gres ** 2).sum() / ((ly[ok] - ly[ok].mean()) ** 2).sum():.4f}; residual sd {gres.std():.3f}; test supported {sup.sum()}")

GP out-of-fold R2 0.8385; residual sd 0.658; test supported 19386


In [7]:
for path in ("v61_q.npz", "v65_q.npz", "v73_q.npz", "v81_q.npz"):
    z_ = np.load(REP / path); RANK.update({k: logit(z_[k]) for k in z_.files})
P = pd.read_csv(REP / "v82all_post.csv"); P = P[P.wt > 1e-5].copy(); P.wt /= P.wt.sum()
print(f"{len(P)} hypotheses carry the posterior; weight by ranking family:",
      P.assign(f=np.where(P.R.str.startswith("catNO"), "catboost", np.where(P.R.str.startswith("aft") | P.R.isin(["clfaft"]) | P.R2.isin(["aft", "aftd6", "clfaft"]), "survival", "classifier"))).groupby("f").wt.sum().round(3).to_dict())
mu_oof = oof[ok]; edges = np.quantile(mu_oof, [1 / 3, 2 / 3])
band_of_failure = np.searchsorted(edges, mu_oof); band = np.searchsorted(edges, np.where(sup, sev, 0.0))
QB = np.stack([np.quantile(gres[band_of_failure == j], grid) for j in range(3)])
draws = np.where(sup[:, None], np.exp(np.where(sup, sev, m)[:, None] + QB[band]), np.exp(m[:, None] + np.quantile(resid, grid)[None, :]))
fl, bu = bayes_decide(*posterior_means(P), draws)
print(f"{int(fl.sum())} flags, mean budget {bu[fl == 1].mean():.0f}")
write_submission(te.id, fl, bu, "v82_hedge")

6861 hypotheses carry the posterior; weight by ranking family: {'catboost': 0.076, 'classifier': 0.706, 'survival': 0.218}


1436 flags, mean budget 634
wrote outputs/submission_v82_hedge.csv: flagged 7.18% of test vehicles


**Evidence.** Under every ranking family the posterior expects this file within about -42 to +14 of v51 on the public 30% (-11 overall). The public difference between two files this close has noise of about ±300.